# Sub-Clinical Detection, Early-Abort, and Cross-Model Transfer

Three related tests, bundled into one session because they all reuse the same natural-pool +
classifier machinery from `08-ai4dd-scaled-final-validation.ipynb`, which has to be rebuilt from
scratch here -- nothing from `08` was ever saved as a reusable model object.

## What this answers

**1. Sub-clinical detection (the idea that ties both arms of the paper together).** At 1x steering
strength, the outcome metric (collapse rate) shows no significant change from CONTROL
(`notes/locked-results.md` §1a/§1g/§1l) -- by that measure, 1x steering looks harmless. But does the
*collapse predictor* -- trained only on natural, unsteered data -- already see something the crude
outcome measure misses? If predicted-collapse probability is elevated in 1x-steered sequences even
though realized collapse isn't, that is real internal damage the endpoint metric is blind to, and it
makes the predictor more sensitive than the thing it was trained to predict.

**2. Cross-model transfer.** ProtGPT2 and ZymCTRL are both 1280-dim, 36-layer, same architecture
family. Train the classifier on ProtGPT2 only, then apply it *unmodified* -- no retraining, no
refitting -- to ZymCTRL's own natural pool. If it still predicts ZymCTRL's collapse better than
chance, folding failure has a shared representational signature across models, not a
ProtGPT2-specific one.

**3. Early-abort prediction.** `08`'s classifier reads the *entire* generated sequence. Re-run it
using only the first 10, 20, or 30 residues of each generation (encoded via the same teacher-forced
method, just truncated). If AUC holds up at 20 residues, a doomed generation could be aborted before
it finishes and before it ever reaches ESMFold -- a real, well-founded early-warning system, and the
defensible version of the abstention-gate idea this project shelved back in August.

## Design decisions, stated up front

- **N=250** for the natural training pool, not `08`'s N=400 -- a deliberate scope-down given the
  time budget; this is a new methodological test, not a replication of `08`'s own number, so it
  doesn't need to match that N exactly.
- **Layer 30 only**, the best single layer from `08`'s own sweep (§3a). No multi-layer sweep here --
  the question this notebook asks is orthogonal to which layer is best, and re-sweeping would only
  add cost.
- **RandomForest only**, not RF+SVM. `08`'s own numbers show them performing similarly; picking one
  halves the classifier-fitting cost across every repeated split.
- **`n_repeats=15`**, not `08`'s 20 -- close enough for a stable AUC estimate, faster.
- A **single "production" classifier** is fit once on the *entire* N=250 pool (all of it, no
  held-out split) for the sub-clinical and transfer tests -- this is standard practice for a
  deployable model, and it is then evaluated only on genuinely new sequences (freshly generated
  CONTROL/1x-steered sets, and ZymCTRL's separate pool) that were never part of its training data,
  so there is no leakage.
- Reuses the length-aware ESMFold evaluator (cap + one retry) from `39`/`40`/`41`, and the naive
  `v_L` construction from `03`/`35`/`37` for the 1x steering condition, at ProtGPT2 layer 12 --
  matching the project's most-cited operating point so the result is directly comparable to
  locked §1a.

Kaggle setup: Accelerator = **GPU T4 x1 or x2**, Internet = **ON**. Expect ~2.5-3 hours
(250 pool folds + ~100 sub-clinical folds + ~120 ZymCTRL transfer folds, plus repeated classifier
fits at 4 truncation lengths).


In [1]:
import warnings
warnings.filterwarnings("ignore")

import gc
import math
import collections
import urllib.request

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, EsmForProteinFolding
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score
from scipy.stats import mannwhitneyu, fisher_exact

torch.manual_seed(2024)
np.random.seed(2024)

device = "cuda" if torch.cuda.is_available() else "cpu"
REFERENCE_NORM = 583.998
TARGET_LAYER_INJECT = 12    # where the steering vector is injected (03/24's convention)
TARGET_LAYER_READ = 30      # where features are read for prediction (08's best layer, §3a)
N_POOL = 250
TRUNC_LENGTHS = [10, 20, 30, None]   # None = full sequence

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def calculate_entropy(seq_str):
    if not seq_str:
        return 0.0
    counts = collections.Counter(seq_str)
    total = len(seq_str)
    return -sum((c / total) * math.log2(c / total) for c in counts.values())

print("Setup complete. CUDA available:", torch.cuda.is_available())


Setup complete. CUDA available: True


In [2]:
# --- Common probe / prefix infrastructure, same fallback pattern as every notebook since 34. ---
UNIPROT_ACCESSIONS = [
    "P0CG48", "P00720", "P02144", "P42212", "P01308", "P61823",
    "P00648", "P99999", "P69905", "P68871", "P00698", "P00441",
]

def fetch_uniprot_sequence(accession, timeout=10):
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.fasta"
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            text = resp.read().decode("utf-8")
        lines = [l for l in text.strip().split("\n") if l]
        seq = "".join(lines[1:])
        return seq if len(seq) >= 20 else None
    except Exception as e:
        print(f"  skip {accession}: {e}")
        return None

FALLBACK_SEQS = [
    "NLYIQWLKDGGPSSGRPPPS",
    "LSDEDFKAVFGMTRSAFANLPLWKQQHLKKEKGLF",
    "GSQIGAKNTGQVQLNLLAL",
    "MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE",
    "MKTIIALSYIFCLVFADYKDDDDKLEHTHHHEASGGNLQVQLQESGGGLVQAGGSLRLSCAASGRTFSNYAMGWFRQAPGKEREFVAAISWSGGSTYYTDSVKGRFTISRDNAKNTVYLQMNSLKPEDTAVYYCAASRFRYWGQGTQVTVSS",
    "DEPPQSPWDRVKDFATVYVDAVKPTGKGKV",
]

print("Fetching real reference protein sequences from UniProt...")
reference_seqs = []
for acc in UNIPROT_ACCESSIONS:
    seq = fetch_uniprot_sequence(acc)
    if seq:
        reference_seqs.append((acc, seq))
        print(f"  fetched {acc}: {len(seq)} residues")

USED_FALLBACK = False
if not reference_seqs:
    USED_FALLBACK = True
    print("!" * 78)
    print("!! UniProt fetch returned NOTHING -- Kaggle's Internet toggle is likely OFF.")
    print("!! Fix: Notebook sidebar -> Session options -> Internet -> ON, then re-run.")
    print("!" * 78)
    reference_seqs = [(f"local{i + 1}", s) for i, s in enumerate(FALLBACK_SEQS)]

def build_prefix_pool(reference_seqs, n_prefixes, min_len=10, max_len=15, seed=11):
    if not reference_seqs:
        raise RuntimeError("No reference sequences available -- check Internet settings.")
    rng = np.random.RandomState(seed)
    prefixes = []
    for i in range(n_prefixes):
        acc, seq = reference_seqs[i % len(reference_seqs)]
        plen = rng.randint(min_len, max_len + 1)
        start = rng.randint(0, max(1, len(seq) - plen))
        prefixes.append(seq[start:start + plen])
    return prefixes

print(f"\n{len(reference_seqs)} source proteins available for prefix generation.")


Fetching real reference protein sequences from UniProt...
  fetched P0CG48: 685 residues
  fetched P00720: 164 residues
  fetched P02144: 154 residues
  fetched P42212: 238 residues
  fetched P01308: 110 residues
  fetched P61823: 150 residues
  fetched P00648: 157 residues
  fetched P99999: 105 residues
  fetched P69905: 142 residues
  fetched P68871: 147 residues
  fetched P00698: 147 residues
  fetched P00441: 154 residues

12 source proteins available for prefix generation.


In [3]:
# --- Length-aware ESMFold evaluator, same as 39/40/41. ---
VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

class SafeStructuralEvaluator:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print("Loading ESMFold...")
        self.tokenizer = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
        self.model = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1", low_cpu_mem_usage=True)
        self.model = self.model.to(self.device).eval()

    def fold_one(self, seq, max_len=300):
        cleaned = "".join(a for a in seq if a in VALID_AA)
        if len(cleaned) < 10:
            return 0.0, 0.0, False
        working = cleaned[:max_len] if len(cleaned) > max_len else cleaned
        try:
            inputs = self.tokenizer([working], return_tensors="pt", add_special_tokens=False).to(self.device)
            with torch.no_grad():
                out = self.model(**inputs)
            raw_plddt = float(np.mean(out.plddt.cpu().numpy()))
            plddt = raw_plddt * 100.0 if raw_plddt <= 1.5 else raw_plddt
            ptm = float(out.ptm.item()) if hasattr(out, "ptm") else 0.0
            return plddt, ptm, True
        except RuntimeError:
            clear_gpu()
        half = max(10, len(working) // 2)
        if half < len(working):
            try:
                inputs = self.tokenizer([working[:half]], return_tensors="pt", add_special_tokens=False).to(self.device)
                with torch.no_grad():
                    out = self.model(**inputs)
                raw_plddt = float(np.mean(out.plddt.cpu().numpy()))
                plddt = raw_plddt * 100.0 if raw_plddt <= 1.5 else raw_plddt
                ptm = float(out.ptm.item()) if hasattr(out, "ptm") else 0.0
                return plddt, ptm, True
            except RuntimeError:
                clear_gpu()
        return 0.0, 0.0, False

def fold_batch(records, evaluator, key="sequence"):
    for r in records:
        plddt, ptm, fold_ok = evaluator.fold_one(r[key])
        r["plddt"] = plddt
        r["ptm"] = ptm
        r["fold_ok"] = fold_ok
        r["collapse"] = int(0.0 < plddt < 60.0)
    return records

print("Length-aware ESMFold evaluator ready.")


Length-aware ESMFold evaluator ready.


In [4]:
# --- Step 1: the natural training pool. Plain ProtGPT2 generation, same recipe as 24/36. ---

print(f"Loading ProtGPT2 on {device}...")
tokenizer = AutoTokenizer.from_pretrained("nferruz/ProtGPT2")
plm_model = AutoModelForCausalLM.from_pretrained("nferruz/ProtGPT2").to(device)
plm_model.eval()

def generate_natural(tokenizer, model, prompts, max_len=50, seed=0):
    torch.manual_seed(seed)
    records = []
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs, max_length=max_len, do_sample=True,
                temperature=1.2, pad_token_id=tokenizer.eos_token_id
            )
        seq = tokenizer.decode(output_ids[0], skip_special_tokens=True).replace(" ", "")
        gen_only = seq[len(prompt):] if seq.startswith(prompt) else seq
        records.append({"prompt": prompt, "sequence": seq, "gen_only": gen_only})
    clear_gpu()
    return records

pool_prefixes = build_prefix_pool(reference_seqs, n_prefixes=N_POOL, seed=808)
print(f"=== Generating N={N_POOL} natural training-pool sequences ===")
pool_records = generate_natural(tokenizer, plm_model, pool_prefixes, max_len=50, seed=808)

print("=== Freeing ProtGPT2 while ESMFold folds the pool ===")
del plm_model
clear_gpu()

evaluator = SafeStructuralEvaluator()
print("Folding the natural training pool...")
pool_records = fold_batch(pool_records, evaluator, key="sequence")
del evaluator
clear_gpu()

n_ok = sum(1 for r in pool_records if r["fold_ok"])
collapse_rate = np.mean([r["collapse"] for r in pool_records])
print(f"\n{n_ok}/{len(pool_records)} pool sequences folded successfully.")
print(f"Natural collapse rate: {collapse_rate:.1%}")
print(f"(08 recorded 53.2% at N=400 -- a reasonably close value here confirms this pool is")
print(f" comparable, even at a different N and a different random draw.)")


Loading ProtGPT2 on cuda...


config.json:   0%|          | 0.00/850 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/4533 [00:00<?, ?it/s]

EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.weight | MISSING    | 
esm.contact_head.regression.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Folding the natural training pool...

250/250 pool sequences folded successfully.
Natural collapse rate: 56.4%
(08 recorded 53.2% at N=400 -- a reasonably close value here confirms this pool is
 comparable, even at a different N and a different random draw.)


In [5]:
# --- Step 2: extract features at layer 30, at every truncation length, via teacher-forced
#     re-encoding -- 08's own method (a forward pass with output_hidden_states=True, not the
#     generation call itself). Truncation is applied to the GENERATED portion only (gen_only),
#     with the original prompt kept intact, since that mirrors what "abort after K residues of
#     generation" would actually see. ---

print(f"Reloading ProtGPT2 on {device} to extract features...")
plm_model = AutoModelForCausalLM.from_pretrained("nferruz/ProtGPT2").to(device)
plm_model.eval()

def extract_layer_feature(model, tokenizer, prompt, gen_only, layer, trunc_len=None):
    gen_part = gen_only if trunc_len is None else gen_only[:trunc_len]
    full_seq = prompt + gen_part
    if len(full_seq) < 5:
        return None
    inputs = tokenizer(full_seq, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True)
        feat = out.hidden_states[layer].mean(dim=1).squeeze(0).cpu().numpy()
    return feat

print(f"Extracting layer-{TARGET_LAYER_READ} features at truncation lengths {TRUNC_LENGTHS}...")
features_by_trunc = {t: [] for t in TRUNC_LENGTHS}
labels_by_trunc = {t: [] for t in TRUNC_LENGTHS}
valid_pool = [r for r in pool_records if r["fold_ok"]]

for t in TRUNC_LENGTHS:
    for r in valid_pool:
        feat = extract_layer_feature(plm_model, tokenizer, r["prompt"], r["gen_only"],
                                     TARGET_LAYER_READ, trunc_len=t)
        if feat is not None:
            features_by_trunc[t].append(feat)
            labels_by_trunc[t].append(r["collapse"])
    print(f"  trunc={t}: {len(features_by_trunc[t])} feature vectors extracted")

X_full = np.array(features_by_trunc[None])
y_full = np.array(labels_by_trunc[None])
print(f"\nFull-length feature matrix: {X_full.shape}, collapse rate {y_full.mean():.1%}")


Reloading ProtGPT2 on cuda to extract features...


Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: nferruz/ProtGPT2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extracting layer-30 features at truncation lengths [10, 20, 30, None]...
  trunc=10: 250 feature vectors extracted
  trunc=20: 250 feature vectors extracted
  trunc=30: 250 feature vectors extracted
  trunc=None: 250 feature vectors extracted

Full-length feature matrix: (250, 1280), collapse rate 56.4%


In [6]:
# --- Step 3 (item 7): early-abort -- repeated-split ROC-AUC at each truncation length. ---

def fit_eval(X, y, n_repeats=15, test_size=0.3, seed=99):
    aucs = []
    sss = StratifiedShuffleSplit(n_splits=n_repeats, test_size=test_size, random_state=seed)
    for train_idx, test_idx in sss.split(X, y):
        y_train, y_test = y[train_idx], y[test_idx]
        if len(set(y_train)) < 2 or len(set(y_test)) < 2:
            continue
        X_train, X_test = X[train_idx], X[test_idx]
        scaler = StandardScaler().fit(X_train)
        X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)
        n_comp = min(20, X_train_s.shape[0] - 1, X_train_s.shape[1])
        pca = PCA(n_components=n_comp, random_state=seed).fit(X_train_s)
        X_train_p, X_test_p = pca.transform(X_train_s), pca.transform(X_test_s)
        clf = RandomForestClassifier(n_estimators=200, random_state=seed)
        clf.fit(X_train_p, y_train)
        prob = clf.predict_proba(X_test_p)[:, 1]
        aucs.append(roc_auc_score(y_test, prob))
    return np.array(aucs)

print("=" * 78)
print("EARLY-ABORT: ROC-AUC by how many residues the classifier is allowed to see")
print("=" * 78)
print(f"{'Truncation':>12s} {'N':>5s} {'mean AUC':>10s} {'std':>8s}")
print("-" * 45)
auc_by_trunc = {}
for t in TRUNC_LENGTHS:
    X = np.array(features_by_trunc[t])
    y = np.array(labels_by_trunc[t])
    aucs = fit_eval(X, y)
    auc_by_trunc[t] = aucs
    label = f"{t} residues" if t is not None else "full sequence"
    print(f"{label:>12s} {len(y):5d} {aucs.mean():10.3f} {aucs.std():8.3f}")

print()
full_auc = auc_by_trunc[None].mean()
best_short = max((t for t in TRUNC_LENGTHS if t is not None), key=lambda t: auc_by_trunc[t].mean())
best_short_auc = auc_by_trunc[best_short].mean()
retained = best_short_auc / full_auc if full_auc > 0 else float("nan")
print(f"Best short-truncation AUC ({best_short} residues): {best_short_auc:.3f} "
      f"({retained:.0%} of full-sequence AUC of {full_auc:.3f})")
if retained > 0.85:
    print("  ==> Early abort looks viable: most of the predictive signal is already present in")
    print(f"      the first {best_short} residues. This supports a real early-warning system --")
    print("      a doomed generation could be flagged well before it finishes or reaches ESMFold.")
else:
    print("  ==> Most of the predictive signal only appears late in generation. Early abort is not")
    print("      well-supported by this data -- report as a negative/limiting result, not a win.")


EARLY-ABORT: ROC-AUC by how many residues the classifier is allowed to see
  Truncation     N   mean AUC      std
---------------------------------------------
 10 residues   250      0.682    0.040
 20 residues   250      0.720    0.036
 30 residues   250      0.740    0.029
full sequence   250      0.780    0.037

Best short-truncation AUC (30 residues): 0.740 (95% of full-sequence AUC of 0.780)
  ==> Early abort looks viable: most of the predictive signal is already present in
      the first 30 residues. This supports a real early-warning system --
      a doomed generation could be flagged well before it finishes or reaches ESMFold.


In [7]:
# --- Step 4: fit the ONE production classifier on the full N=250 pool (all of it), for use
#     in the sub-clinical and transfer tests below. No held-out split here -- this model will only
#     ever be evaluated on genuinely new sequences generated after this point. ---

scaler_prod = StandardScaler().fit(X_full)
X_full_scaled = scaler_prod.transform(X_full)
n_comp_prod = min(20, X_full_scaled.shape[0] - 1, X_full_scaled.shape[1])
pca_prod = PCA(n_components=n_comp_prod, random_state=99).fit(X_full_scaled)
X_full_pca = pca_prod.transform(X_full_scaled)
clf_prod = RandomForestClassifier(n_estimators=300, random_state=99)
clf_prod.fit(X_full_pca, y_full)

def predict_prob(model, tokenizer, prompt, gen_only, layer=TARGET_LAYER_READ):
    feat = extract_layer_feature(model, tokenizer, prompt, gen_only, layer)
    if feat is None:
        return None
    feat_s = scaler_prod.transform(feat.reshape(1, -1))
    feat_p = pca_prod.transform(feat_s)
    return float(clf_prod.predict_proba(feat_p)[0, 1])

print(f"Production classifier fit on all N={len(y_full)} natural pool sequences.")
print(f"(In-sample check only, not a generalization estimate -- the real estimate is the")
print(f" full-length row of the early-abort table above: {full_auc:.3f})")


Production classifier fit on all N=250 natural pool sequences.
(In-sample check only, not a generalization estimate -- the real estimate is the
 full-length row of the early-abort table above: 0.780)


In [8]:
# --- Step 5 (item 5): sub-clinical detection. Generate FRESH CONTROL and REAL_1x sequences
#     (naive v_L, layer 12 injection -- same construction as 03/35/37), never seen by the
#     classifier, and check whether predicted-collapse probability shifts even though realized
#     collapse doesn't (per locked §1a/§1l: no significant difference at 1x). ---

positive_seqs = [
    "NLYIQWLKDGGPSSGRPPPS", "LSDEDFKAVFGMTRSAFANLPLWKQQHLKKEKGLF", "GSQIGAKNTGQVQLNLLAL",
    "MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE",
    "MKTIIALSYIFCLVFADYKDDDDKLEHTHHHEASGGNLQVQLQESGGGLVQAGGSLRLSCAASGRTFSNYAMGWFRQAPGKEREFVAAISWSGGSTYYTDSVKGRFTISRDNAKNTVYLQMNSLKPEDTAVYYCAASRFRYWGQGTQVTVSS",
    "DEPPQSPWDRVKDFATVYVDAVKPTGKGKV",
]
degenerate_seqs = [
    "AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA", "LGLGLGLGLGLGLGLGLGLGLGLGLGLGLGLGLGL",
    "GGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGG", "SSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSS",
    "PGPGPGPGPGPGPGPGPGPGPGPGPGPGPGPGPGP", "QWQWQWQWQWQWQWQWQWQWQWQWQWQWQWQWQWQ",
]

def get_mean_activation(model, tokenizer, seq_list, layer):
    acts = []
    for seq in seq_list:
        inputs = tokenizer(seq, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model(**inputs, output_hidden_states=True)
            acts.append(out.hidden_states[layer].mean(dim=1).squeeze(0).cpu())
    return torch.stack(acts)

pos_acts = get_mean_activation(plm_model, tokenizer, positive_seqs, TARGET_LAYER_INJECT)
neg_acts = get_mean_activation(plm_model, tokenizer, degenerate_seqs, TARGET_LAYER_INJECT)
v_L_raw = pos_acts.mean(dim=0) - neg_acts.mean(dim=0)
v_L = (v_L_raw * (REFERENCE_NORM / v_L_raw.norm().item())).to(device)
print(f"Naive v_L norm: {v_L.norm().item():.4f} (03/35/37 all report ~583.998 -- confirms faithful rebuild)")

def generate_with_vector_steering(model, tokenizer, target_layer, steering_vector, prompts, max_len=50, seed=0):
    torch.manual_seed(seed)
    model.eval()
    v = None if steering_vector is None else steering_vector.to(device)

    def hook(module, inp, out):
        if v is None:
            return out
        return (out[0] + v,)

    records = []
    for prompt in prompts:
        handle = model.transformer.h[target_layer].register_forward_hook(hook)
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs, max_length=max_len, do_sample=True,
                temperature=1.2, pad_token_id=tokenizer.eos_token_id
            )
        handle.remove()
        seq = tokenizer.decode(output_ids[0], skip_special_tokens=True).replace(" ", "")
        gen_part = seq[len(prompt):] if seq.startswith(prompt) else seq
        records.append({"prompt": prompt, "sequence": seq, "gen_only": gen_part})
    clear_gpu()
    return records

subclin_prefixes = build_prefix_pool(reference_seqs, n_prefixes=100, seed=909)
N_SUBCLIN = 50

print("=== CONTROL (fresh, unseen by classifier) ===")
subclin_control = generate_with_vector_steering(
    plm_model, tokenizer, TARGET_LAYER_INJECT, None, subclin_prefixes[:N_SUBCLIN], seed=1111)
print("=== REAL_1x (naive v_L, layer 12, reference strength) ===")
subclin_1x = generate_with_vector_steering(
    plm_model, tokenizer, TARGET_LAYER_INJECT, v_L * 1.0, subclin_prefixes[N_SUBCLIN:2*N_SUBCLIN], seed=2222)

print("\n=== Freeing ProtGPT2 (candidate-pool and steering generation both done) ===")
del plm_model
clear_gpu()


Naive v_L norm: 583.9980 (03/35/37 all report ~583.998 -- confirms faithful rebuild)
=== CONTROL (fresh, unseen by classifier) ===
=== REAL_1x (naive v_L, layer 12, reference strength) ===

=== Freeing ProtGPT2 (candidate-pool and steering generation both done) ===


In [9]:
# --- Fold the sub-clinical sets, then get the classifier's predicted probability for each --
#     using the extraction function defined earlier, which needs a live model, so reload once more. ---

evaluator2 = SafeStructuralEvaluator()
print("Folding CONTROL (sub-clinical set)...")
subclin_control = fold_batch(subclin_control, evaluator2, key="sequence")
print("Folding REAL_1x (sub-clinical set)...")
subclin_1x = fold_batch(subclin_1x, evaluator2, key="sequence")
del evaluator2
clear_gpu()

ctrl_k = sum(r["collapse"] for r in subclin_control)
x1_k = sum(r["collapse"] for r in subclin_1x)
print(f"\nRealized collapse -- CONTROL: {ctrl_k}/{len(subclin_control)} "
      f"({ctrl_k/len(subclin_control):.1%}), REAL_1x: {x1_k}/{len(subclin_1x)} "
      f"({x1_k/len(subclin_1x):.1%})")
_, p_realized = fisher_exact([[x1_k, len(subclin_1x) - x1_k], [ctrl_k, len(subclin_control) - ctrl_k]])
print(f"Fisher p (realized collapse, 1x vs CONTROL): {p_realized:.4f}")
print("(locked §1a/§1l already established this is not significant at N=50 -- this is a check,")
print(" not a new claim.)")

print(f"\nReloading ProtGPT2 on {device} for the classifier's own feature extraction...")
plm_model = AutoModelForCausalLM.from_pretrained("nferruz/ProtGPT2").to(device)
plm_model.eval()

ctrl_probs = [predict_prob(plm_model, tokenizer, r["prompt"], r["gen_only"]) for r in subclin_control]
x1_probs = [predict_prob(plm_model, tokenizer, r["prompt"], r["gen_only"]) for r in subclin_1x]
ctrl_probs = [p for p in ctrl_probs if p is not None]
x1_probs = [p for p in x1_probs if p is not None]

print("=== Freeing ProtGPT2 (feature extraction for sub-clinical set is done) ===")
del plm_model
clear_gpu()

print(f"\nClassifier's predicted collapse-probability -- CONTROL: mean={np.mean(ctrl_probs):.3f}, "
      f"REAL_1x: mean={np.mean(x1_probs):.3f}")
u_stat, p_prob = mannwhitneyu(x1_probs, ctrl_probs, alternative="greater")
print(f"Mann-Whitney U (is REAL_1x's predicted probability distribution shifted HIGHER than "
      f"CONTROL's?): U={u_stat:.1f}, p={p_prob:.4f}")

print()
print("=" * 90)
print("VERDICT -- sub-clinical detection")
print("=" * 90)
if p_prob < 0.05 and p_realized >= 0.05:
    print("  ==> SUB-CLINICAL DAMAGE DETECTED. Realized collapse does NOT differ (1x looks safe by")
    print("      the outcome metric), but the classifier's predicted risk IS significantly higher")
    print("      for 1x-steered sequences. This is the headline result this notebook was built to")
    print("      find: steering at 'reference strength' does measurable internal damage the crude")
    print("      pass/fail metric misses, and the deep-layer classifier is more sensitive than the")
    print("      endpoint it was trained on. This unifies the paper's two arms into one thesis.")
elif p_prob < 0.05 and p_realized < 0.05:
    print("  ==> Both realized collapse AND predicted probability differ at 1x in this specific run")
    print("      -- re-check against the locked §1a/§1l numbers (N=50 noise can occasionally trip")
    print("      significance in either direction); if this replicates, 1x may not be as 'safe' as")
    print("      previously reported and locked-results.md needs a second look, not just this note.")
else:
    print("  ==> No detectable shift in predicted probability either. The classifier does not see")
    print("      anything at 1x that the outcome metric misses -- a real, informative null result,")
    print("      not a failed test. Report as: 'no evidence of sub-clinical damage at reference")
    print("      steering strength, by this classifier.'")

Loading ESMFold...


Loading weights:   0%|          | 0/4533 [00:00<?, ?it/s]

EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.weight | MISSING    | 
esm.contact_head.regression.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Folding CONTROL (sub-clinical set)...
Folding REAL_1x (sub-clinical set)...

Realized collapse -- CONTROL: 31/50 (62.0%), REAL_1x: 39/50 (78.0%)
Fisher p (realized collapse, 1x vs CONTROL): 0.1259
(locked §1a/§1l already established this is not significant at N=50 -- this is a check,
 not a new claim.)

Reloading ProtGPT2 on cuda for the classifier's own feature extraction...


Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: nferruz/ProtGPT2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== Freeing ProtGPT2 (feature extraction for sub-clinical set is done) ===

Classifier's predicted collapse-probability -- CONTROL: mean=0.606, REAL_1x: mean=0.654
Mann-Whitney U (is REAL_1x's predicted probability distribution shifted HIGHER than CONTROL's?): U=1398.5, p=0.1538

VERDICT -- sub-clinical detection
  ==> No detectable shift in predicted probability either. The classifier does not see
      anything at 1x that the outcome metric misses -- a real, informative null result,
      not a failed test. Report as: 'no evidence of sub-clinical damage at reference
      steering strength, by this classifier.'


In [10]:
# --- Step 6 (transfer): apply the SAME classifier (no retraining) to ZymCTRL's own natural
#     pool. Both models are 1280-dim/36-layer GPT-2-family, so no architecture adaptation is
#     needed -- the whole point is testing whether the classifier trained on ProtGPT2 features
#     means anything on ZymCTRL's. ---

print(f"Loading ZymCTRL on {device} for the transfer test...")
zym_tokenizer = AutoTokenizer.from_pretrained("AI4PD/ZymCTRL")
zym_model = AutoModelForCausalLM.from_pretrained("AI4PD/ZymCTRL").to(device)
zym_model.eval()
ZYM_EOS = zym_tokenizer.eos_token_id if zym_tokenizer.eos_token_id is not None else 1
ZYM_PAD = zym_tokenizer.pad_token_id if zym_tokenizer.pad_token_id is not None else ZYM_EOS

EC_LABELS = [
    "1.1.1.1", "1.1.1.2", "2.7.1.1", "2.7.1.2", "3.1.1.1", "3.5.1.4",
    "4.1.1.1", "4.2.1.1", "5.1.3.1", "5.3.1.9", "6.1.1.1", "6.3.2.1",
]

def clean_zymctrl_output(decoded_text):
    seq_part = decoded_text.split("<sep>", 1)[1] if "<sep>" in decoded_text else decoded_text
    for tok in ["<start>", "<end>", "<|endoftext|>", "<pad>", " "]:
        seq_part = seq_part.replace(tok, "")
    return seq_part

def build_ec_prompt_pool(labels, n_prompts, seed=11):
    rng = np.random.RandomState(seed)
    order = rng.permutation(n_prompts)
    return [labels[i % len(labels)] for i in order]

def generate_zymctrl_natural(tokenizer, model, prompts, max_len=50, seed=0):
    torch.manual_seed(seed)
    records = []
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs, max_length=max_len, do_sample=True,
                temperature=1.2, eos_token_id=ZYM_EOS, pad_token_id=ZYM_PAD
            )
        raw_decoded = tokenizer.decode(output_ids[0], skip_special_tokens=False)
        seq = clean_zymctrl_output(raw_decoded)
        records.append({"prompt": prompt, "sequence": seq, "gen_only": seq})
    clear_gpu()
    return records

N_ZYM = 120
zym_prompts = build_ec_prompt_pool(EC_LABELS, N_ZYM, seed=3030)
print(f"=== Generating N={N_ZYM} ZymCTRL natural sequences for the transfer test ===")
zym_records = generate_zymctrl_natural(zym_tokenizer, zym_model, zym_prompts, max_len=50, seed=3030)

print("=== Freeing ZymCTRL generation model while ESMFold folds ===")
del zym_model
clear_gpu()

evaluator3 = SafeStructuralEvaluator()
print("Folding ZymCTRL's transfer-test pool...")
zym_records = fold_batch(zym_records, evaluator3, key="sequence")
del evaluator3
clear_gpu()

zym_collapse_rate = np.mean([r["collapse"] for r in zym_records if r["fold_ok"]])
print(f"\nZymCTRL natural collapse rate (this pool): {zym_collapse_rate:.1%}")
print(f"(26/38/40 recorded ~61.5% -- a close value confirms this pool is comparable)")


Loading ZymCTRL on cuda for the transfer test...


config.json:   0%|          | 0.00/765 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.88G [00:00<?, ?B/s]

...

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: AI4PD/ZymCTRL
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== Generating N=120 ZymCTRL natural sequences for the transfer test ===
=== Freeing ZymCTRL generation model while ESMFold folds ===
Loading ESMFold...


Loading weights:   0%|          | 0/4533 [00:00<?, ?it/s]

EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.weight | MISSING    | 
esm.contact_head.regression.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Folding ZymCTRL's transfer-test pool...

ZymCTRL natural collapse rate (this pool): 65.8%
(26/38/40 recorded ~61.5% -- a close value confirms this pool is comparable)


In [11]:
# --- Extract ZymCTRL's own layer-30 features and predict with the PROTGPT2-TRAINED
#     classifier, unmodified -- no refitting of scaler/PCA/RF against ZymCTRL's own data. ---

print(f"Reloading ZymCTRL on {device} to extract layer-{TARGET_LAYER_READ} features...")
zym_model = AutoModelForCausalLM.from_pretrained("AI4PD/ZymCTRL").to(device)
zym_model.eval()

zym_valid = [r for r in zym_records if r["fold_ok"]]
zym_X, zym_y = [], []
for r in zym_valid:
    feat = extract_layer_feature(zym_model, zym_tokenizer, "", r["sequence"], TARGET_LAYER_READ)
    if feat is not None:
        zym_X.append(feat)
        zym_y.append(r["collapse"])
zym_X = np.array(zym_X)
zym_y = np.array(zym_y)

del zym_model
clear_gpu()

zym_X_scaled = scaler_prod.transform(zym_X)   # SAME scaler fit on ProtGPT2, not refit
zym_X_pca = pca_prod.transform(zym_X_scaled)   # SAME PCA fit on ProtGPT2, not refit
zym_probs = clf_prod.predict_proba(zym_X_pca)[:, 1]

print("=" * 90)
print("VERDICT -- cross-model transfer (ProtGPT2-trained classifier, applied to ZymCTRL untouched)")
print("=" * 90)
if len(set(zym_y)) < 2:
    print("Cannot compute ROC-AUC -- ZymCTRL's pool has only one class present (all collapsed or")
    print("all survived). Report the raw collapse rate as the only usable number from this test.")
    zym_transfer_auc = float("nan")
else:
    zym_transfer_auc = roc_auc_score(zym_y, zym_probs)
    print(f"Transfer AUC (ProtGPT2-trained classifier -> ZymCTRL's true labels): {zym_transfer_auc:.3f}")
    print(f"(0.5 = chance, and the classifier saw zero ZymCTRL data during training)")
    print()
    if zym_transfer_auc > 0.60:
        print("  ==> REAL TRANSFER. A classifier trained purely on ProtGPT2's own natural")
        print("      generations meaningfully predicts ZymCTRL's folding failures too, with zero")
        print("      ZymCTRL-specific training. This means fold-failure has a shared")
        print("      representational signature across at least these two models -- a striking")
        print("      result given how different EC-conditioned generation is from plain")
        print("      continuation, and one this project did not have before this notebook.")
    elif zym_transfer_auc > 0.5:
        print("  ==> WEAK/PARTIAL TRANSFER. Better than chance but modest. Report honestly as a")
        print("      real but limited signal -- the representational basis for fold-failure")
        print("      prediction may be only partially shared across models.")
    else:
        print("  ==> NO TRANSFER. The classifier does not generalize to ZymCTRL at all. This is")
        print("      itself informative: whatever the layer-30 classifier is picking up on in")
        print("      ProtGPT2 may be model-specific, not a universal fold-failure signature.")
        print("      ZymCTRL's own separately-trained classifier (locked §3d, AUC 0.741) is real")
        print("      and unaffected by this -- only the CROSS-model transfer claim is negative.")


Reloading ZymCTRL on cuda to extract layer-30 features...


Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: AI4PD/ZymCTRL
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


VERDICT -- cross-model transfer (ProtGPT2-trained classifier, applied to ZymCTRL untouched)
Transfer AUC (ProtGPT2-trained classifier -> ZymCTRL's true labels): 0.409
(0.5 = chance, and the classifier saw zero ZymCTRL data during training)

  ==> NO TRANSFER. The classifier does not generalize to ZymCTRL at all. This is
      itself informative: whatever the layer-30 classifier is picking up on in
      ProtGPT2 may be model-specific, not a universal fold-failure signature.
      ZymCTRL's own separately-trained classifier (locked §3d, AUC 0.741) is real
      and unaffected by this -- only the CROSS-model transfer claim is negative.


In [12]:
# --- Persist everything. ---
early_abort_rows = []
for t in TRUNC_LENGTHS:
    label = f"{t}res" if t is not None else "full"
    for a in auc_by_trunc[t]:
        early_abort_rows.append({"truncation": label, "trunc_len": t if t is not None else -1, "auc": a})
pd.DataFrame(early_abort_rows).to_csv("early_abort_auc_by_truncation.csv", index=False)

subclin_rows = []
for cond, recs, probs in [("CONTROL", subclin_control, ctrl_probs), ("REAL_1x", subclin_1x, x1_probs)]:
    for r, p in zip(recs, probs if len(probs) == len(recs) else [None]*len(recs)):
        subclin_rows.append({
            "condition": cond, "prompt": r["prompt"], "sequence": r["sequence"],
            "plddt": r["plddt"], "collapse": r["collapse"], "predicted_prob": p,
        })
pd.DataFrame(subclin_rows).to_csv("subclinical_detection_sequences.csv", index=False)

zym_rows = [{
    "sequence": r["sequence"], "plddt": r["plddt"], "collapse": r["collapse"], "predicted_prob": p,
} for r, p in zip(zym_valid, zym_probs)]
pd.DataFrame(zym_rows).to_csv("crossmodel_transfer_zymctrl_sequences.csv", index=False)

pd.DataFrame([{
    "n_pool": len(y_full), "pool_collapse_rate": float(y_full.mean()),
    "full_length_auc": full_auc, "best_short_truncation": best_short,
    "best_short_auc": best_short_auc, "auc_retained_fraction": retained,
    "subclin_p_realized": p_realized, "subclin_p_predicted_prob": p_prob,
    "zym_transfer_auc": zym_transfer_auc, "zym_pool_collapse_rate": float(zym_collapse_rate),
}]).to_csv("subclinical_early_abort_transfer_summary.csv", index=False)

print("Saved:")
print("  early_abort_auc_by_truncation.csv")
print("  subclinical_detection_sequences.csv")
print("  crossmodel_transfer_zymctrl_sequences.csv")
print("  subclinical_early_abort_transfer_summary.csv")
print()
print("Update notes/locked-results.md with a new SS5 (or similar) section for whichever of these")
print("three results turned out real -- all three are genuine, reportable outcomes regardless of")
print("which way they landed; see this notebook's own verdict cells for the specific reasoning.")


Saved:
  early_abort_auc_by_truncation.csv
  subclinical_detection_sequences.csv
  crossmodel_transfer_zymctrl_sequences.csv
  subclinical_early_abort_transfer_summary.csv

Update notes/locked-results.md with a new SS5 (or similar) section for whichever of these
three results turned out real -- all three are genuine, reportable outcomes regardless of
which way they landed; see this notebook's own verdict cells for the specific reasoning.
